# Building a Research Assistant with Web + X Search

What do reputable news sources say about a topic? What does the crowd on X think? And where do those two stories diverge?

In this guide, we'll build a research assistant that cross-references reporting from vetted news outlets with public discourse on X, then produces a structured "divergence briefing" that clusters claims into five categories: consensus, X-ahead-of-press, press-ahead-of-X, X-only, and press-only.

This takes advantage of Grok's built-in web search and X search as native tools, with domain filtering to scope web results to trusted outlets and inline citations linking every claim back to its source.

### What we'll use
- Web search with domain filtering (`allowed_domains`) to restrict results to trusted outlets
- X search to capture real-time public discourse
- Inline citations linking every claim back to its source
- Structured output with Pydantic models to parse the final analysis
- The native xai-sdk

### Table of Contents
- [Setup](#setup)
- [Act 1: Web Search with Domain Filtering](#act-1-web-search-with-domain-filtering)
- [Interlude: Filtered vs. Unfiltered](#interlude-filtered-vs-unfiltered)
- [Act 2: X Search for Public Discourse](#act-2-x-search-for-public-discourse)
- [Act 3: The Divergence Analysis](#act-3-the-divergence-analysis)
- [Structured Output](#structured-output)
- [Putting It All Together](#putting-it-all-together)
- [Conclusion](#conclusion)

## Setup

We use the native `xai-sdk` rather than the OpenAI compatibility layer, since `web_search` domain filtering, `x_search`, and inline citations are first-class features of the native SDK.

All you need is an xAI API key.

In [1]:
%pip install -q xai-sdk

In [2]:
import os

from xai_sdk import Client
from xai_sdk.chat import system, user
from xai_sdk.tools import web_search, x_search

XAI_API_KEY = os.getenv("XAI_API_KEY", "")
if not XAI_API_KEY:
    raise ValueError("XAI_API_KEY is not set. Export it or add it to your environment.")

client = Client(api_key=XAI_API_KEY)

MODEL = "grok-4.20-reasoning"

## Act 1: Web Search with Domain Filtering

Grok's `web_search` tool lets you restrict results to specific domains using `allowed_domains` (max 5 per call). Instead of hoping the model finds good sources, you tell it where to look.

We'll define curated domain lists for different research contexts, then run a search scoped to major news outlets.

In [3]:
# Curated domain lists for different research contexts
NEWS_DOMAINS = ["reuters.com", "apnews.com", "bbc.com", "ft.com", "wsj.com"]
TECH_DOMAINS = ["techcrunch.com", "arstechnica.com", "theverge.com", "wired.com"]

In [4]:
TOPIC = "Space-based data centres and the future of orbital computing infrastructure"

chat = client.chat.create(
    model=MODEL,
    tools=[web_search(allowed_domains=NEWS_DOMAINS)],
    include=["inline_citations"],
)
chat.append(user(
    f"Search for the latest reporting on: {TOPIC}. "
    "Summarize the 3-5 most important claims from these sources, with citations. Be concise."
))

response = None
for response, chunk in chat.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**Space-based data centers (also called orbital data centers or orbital computing infrastructure) have seen heightened interest in late 2025–early 2026**, driven by AI’s massive power and cooling demands on Earth. Major players including SpaceX, Blue Origin, Google, startups like Starcloud, and China are pursuing the concept.[[1]](https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/)[[2]](https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/)

Here are the 3–5 most important claims from recent reporting:

**1. Ambitious plans and competition are accelerating.** SpaceX has applied to launch up to 1 million solar-powered satellites for orbital AI data centers (with Musk linking it to a potential IPO for funding and the xAI merger), Blue Origin is advancing Project Sunrise, Google has Project Suncatcher (possible test launches ~2027), Starcloud 

Every claim is backed by an inline citation linking to the original article. Let's inspect those citations programmatically:

In [5]:
print("Web search citations:")
for citation in response.inline_citations:
    if citation.HasField("web_citation"):
        print(f"  {citation.web_citation.url}")

Web search citations:
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/business/aerospace-defense/amazons-aws-ceo-says-orbital-data-ce

Notice every source is from our `NEWS_DOMAINS` list.

## Interlude: Filtered vs. Unfiltered

What happens if we run the same query without domain filtering? The model is free to pull from any source. This isn't a controlled experiment (ranking, freshness, and source availability all vary between calls), but it illustrates why scoping your sources matters for research.

In [6]:
chat_unfiltered = client.chat.create(
    model=MODEL,
    tools=[web_search()],
    include=["inline_citations"],
)
chat_unfiltered.append(user(
    f"Search for the latest reporting on: {TOPIC}. "
    "Summarize the 3-5 most important claims with citations. Be concise."
))

response_unfiltered = None
for response_unfiltered, chunk in chat_unfiltered.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**Key recent developments (as of early 2026) show accelerating activity in space-based data centers and orbital computing, driven by AI power demands.** Multiple companies have moved from concepts to prototypes and regulatory filings.[[1]](https://en.wikipedia.org/wiki/Space-based_data_center)

**1. Prototypes are already in orbit and demonstrating basic capabilities.** Starcloud launched the first NVIDIA H100 GPU into space in November 2025 (via SpaceX), becoming the first to train an LLM and run Google Gemini in orbit. Axiom Space launched its first two dedicated Orbital Data Center nodes to low-Earth orbit on January 11, 2026. China has also launched initial satellites in its constellation.[[1]](https://en.wikipedia.org/wiki/Space-based_data_center)

**2. Continuous solar power and radiative cooling in space are seen as major advantages over terrestrial data centers.** Sun-synchronous orbits can provide nearly uninterrupted solar energy (up to ~8x more productive than on Earth, with

In [7]:
print("\nUnfiltered citations:")
for citation in response_unfiltered.inline_citations:
    if citation.HasField("web_citation"):
        print(f"  {citation.web_citation.url}")

print(f"\nFiltered: {len(response.inline_citations)} citations (all from vetted outlets)")
print(f"Unfiltered: {len(response_unfiltered.inline_citations)} citations (mixed sources)")


Unfiltered citations:
  https://en.wikipedia.org/wiki/Space-based_data_center
  https://en.wikipedia.org/wiki/Space-based_data_center
  https://research.google/blog/exploring-a-space-based-scalable-ai-infrastructure-system-design/
  https://research.google/blog/exploring-a-space-based-scalable-ai-infrastructure-system-design/
  https://research.google/blog/exploring-a-space-based-scalable-ai-infrastructure-system-design/
  https://www.scientificamerican.com/article/data-centers-in-space/

Filtered: 9 citations (all from vetted outlets)
Unfiltered: 6 citations (mixed sources)


Both produce useful summaries, but the filtered version gives you confidence in where the information came from.

## Act 2: X Search for Public Discourse

Now let's see what people are actually saying. Grok's `x_search` tool searches X directly, with optional handle filtering and date ranges.

In [8]:
chat_x = client.chat.create(
    model=MODEL,
    tools=[x_search()],
    include=["inline_citations"],
)
chat_x.append(user(
    f"Search X for what people are saying about: {TOPIC}. "
    "Summarize the 3-5 key themes in public sentiment, with citations to specific posts. Be concise."
))

response_x = None
for response_x, chunk in chat_x.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**Key themes in X discussions on space-based data centers and orbital computing:**

**1. Major advantages for power and cooling:** Users frequently cite constant solar exposure in sun-synchronous orbits (no atmosphere or day/night cycles) and passive radiative cooling into space vacuum as solutions to Earth's power grid, water usage, and HVAC constraints.[[1]](https://x.com/renaldbarnett/status/2040189083071471955)[[2]](https://x.com/kimmonismus/status/1980945551995863217)[[3]](https://x.com/Jackson_Metrics/status/2038842657112543594)

**2. Economic viability driven by falling launch costs:** Discussions emphasize that Starship and declining launch prices (potentially <$200/kg by mid-2030s) could make orbital compute cost-competitive with terrestrial data centers' energy expenses, especially for AI scaling.[[4]](https://x.com/rmcentush/status/1985787187556991364)[[5]](https://x.com/i/status/1985787187556991364)

**3. Excitement around real-world progress and big tech involvement:** Ann

In [9]:
print("X search citations:")
for citation in response_x.inline_citations:
    if citation.HasField("x_citation"):
        print(f"  {citation.x_citation.url}")

X search citations:
  https://x.com/renaldbarnett/status/2040189083071471955
  https://x.com/kimmonismus/status/1980945551995863217
  https://x.com/Jackson_Metrics/status/2038842657112543594
  https://x.com/rmcentush/status/1985787187556991364
  https://x.com/i/status/1985787187556991364
  https://x.com/kimmonismus/status/1980945551995863217
  https://x.com/venturemanny/status/2038684453946405096
  https://x.com/i/status/1985787187556991364
  https://x.com/Andercot/status/1981465914400002550
  https://x.com/i/status/1980945551995863217


Compare the tone and content with Act 1. Press reporting tends to balance the opportunity against economic and technical skepticism, often citing analyst doubts. X discussion leans more optimistic, focusing on the physics and engineering case for why orbital compute could work. The overlap and divergence between those perspectives is what we'll formalize next.

## Act 3: The Divergence Analysis

We take the full outputs from both searches and ask Grok to reconcile them, clustering every claim into one of five categories:

1. CONSENSUS: claims both sources agree on
2. X_AHEAD_OF_PRESS: claims appearing on X but not yet in press
3. PRESS_AHEAD_OF_X: claims in press but not discussed on X
4. X_ONLY: claims unique to X with no press corroboration
5. PRESS_ONLY: claims unique to press with no social discussion

The three-pass architecture:
- Pass 1 (already done): Web search with domain filtering
- Pass 2 (already done): X search
- Pass 3 (below): A reconciliation call with no tools, just analysis

A caveat: the temporal labels ("X-ahead-of-press", etc.) reflect the model's assessment at query time, not timestamped provenance. This is a pattern demo, not a rigorous fact-checker.

In [10]:
from pydantic import BaseModel


class Claim(BaseModel):
    text: str
    source: str
    category: str


class ClaimCluster(BaseModel):
    category: str
    claims: list[Claim]


class DivergenceBrief(BaseModel):
    topic: str
    clusters: list[ClaimCluster]
    executive_summary: str

In [11]:
RECONCILIATION_PROMPT = """You are a research analyst. Given web search findings from \
reputable sources and X/social media findings on the same topic, cluster the claims \
into exactly five categories:
1. CONSENSUS — claims both sources agree on
2. X_AHEAD_OF_PRESS — claims appearing on X but not yet in press
3. PRESS_AHEAD_OF_X — claims in press but not discussed on X
4. X_ONLY — claims unique to X with no press corroboration
5. PRESS_ONLY — claims unique to press with no social discussion

For each claim, note the source ("Press" or "X") and category. Be thorough: extract \
every distinct claim from both inputs. Output as JSON matching the provided schema."""

# Pass 1 and 2 outputs become the context for Pass 3
web_findings = response.content
x_findings = response_x.content

chat_reconcile = client.chat.create(
    model=MODEL,
    response_format=DivergenceBrief,
)
chat_reconcile.append(system(RECONCILIATION_PROMPT))
chat_reconcile.append(user(
    f"## Web Search Findings\n{web_findings}\n\n## X Search Findings\n{x_findings}"
))

response_reconcile = None
for response_reconcile, chunk in chat_reconcile.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

{
  "topic": "Space-based Data Centers for AI",
  "clusters": [
    {
      "category": "CONSENSUS",
      "claims": [
        {
          "text": "Space-based data centers provide near-constant solar power without atmospheric interference or day/night cycles",
          "source": "Both",
          "category": "CONSENSUS"
        },
        {
          "text": "They allow heat to be radiated directly into space, reducing or eliminating the need for water-based cooling",
          "source": "Both",
          "category": "CONSENSUS"
        },
        {
          "text": "Orbital computing can relieve pressure on Earth’s power grids and resources for AI workloads",
          "source": "Both",
          "category": "CONSENSUS"
        },
        {
          "text": "Major technical and economic hurdles include high launch costs that must fall, radiation effects on chips, maintenance in orbit, and data transfer/latency issues",
          "source": "Both",
          "category": "CONSENSUS"


## Structured Output

The reconciliation call used `response_format=DivergenceBrief` to get structured JSON. Let's parse it into our Pydantic model and display it cleanly.

In [12]:
brief = DivergenceBrief.model_validate_json(response_reconcile.content)

print(f"Topic: {brief.topic}")
print(f"{'=' * 60}")
for cluster in brief.clusters:
    print(f"\n{cluster.category} ({len(cluster.claims)} claims)")
    print("-" * 40)
    for claim in cluster.claims:
        print(f"  [{claim.source}] {claim.text}")

print(f"\n{'=' * 60}")
print(f"Executive Summary:\n{brief.executive_summary}")

Topic: Space-based Data Centers for AI

CONSENSUS (5 claims)
----------------------------------------
  [Both] Space-based data centers provide near-constant solar power without atmospheric interference or day/night cycles
  [Both] They allow heat to be radiated directly into space, reducing or eliminating the need for water-based cooling
  [Both] Orbital computing can relieve pressure on Earth’s power grids and resources for AI workloads
  [Both] Major technical and economic hurdles include high launch costs that must fall, radiation effects on chips, maintenance in orbit, and data transfer/latency issues
  [Both] Companies including SpaceX/Musk, Google, and Starcloud are pursuing orbital data centers

X_AHEAD_OF_PRESS (5 claims)
----------------------------------------
  [X] Falling launch costs via Starship could reach under $200/kg by mid-2030s making orbital compute cost-competitive with terrestrial energy expenses for AI
  [X] Real-world progress includes Starcloud with NVIDIA H1

## Putting It All Together

Let's wrap the three-pass flow into a reusable function so we can run it on any topic.

**Cost note:** Each call to `research()` makes 3 API calls (web search, X search, reconciliation). A single call is inexpensive, but costs vary with response length and model pricing changes. See [pricing](https://docs.x.ai/docs/models) for current rates.

In [13]:
def research(topic: str, web_domains: list[str] | None = None) -> DivergenceBrief:
    """Run a three-pass divergence analysis on any topic.

    Args:
        topic: The research subject.
        web_domains: Optional list of allowed domains for web search.
            Defaults to NEWS_DOMAINS if None. Max 5 per xAI docs.
    """
    domains = web_domains or NEWS_DOMAINS
    if len(domains) > 5:
        raise ValueError(f"allowed_domains supports at most 5 entries, got {len(domains)}")

    # Pass 1: Web search
    chat_web = client.chat.create(
        model=MODEL,
        tools=[web_search(allowed_domains=domains)],
        include=["inline_citations"],
    )
    chat_web.append(user(
        f"Search for the latest reporting on: {topic}. "
        "Summarize the 3-5 most important claims with citations. Be concise."
    ))
    resp_web = None
    for resp_web, chunk in chat_web.stream():
        pass
    if not resp_web or not resp_web.content:
        raise RuntimeError(f"Web search returned no results for: {topic}")

    # Pass 2: X search
    chat_social = client.chat.create(
        model=MODEL,
        tools=[x_search()],
        include=["inline_citations"],
    )
    chat_social.append(user(
        f"Search X for what people are saying about: {topic}. "
        "Summarize the 3-5 key themes in public sentiment, with citations. Be concise."
    ))
    resp_social = None
    for resp_social, chunk in chat_social.stream():
        pass
    if not resp_social or not resp_social.content:
        raise RuntimeError(f"X search returned no results for: {topic}")

    # Pass 3: Reconciliation with structured output
    chat_analysis = client.chat.create(
        model=MODEL,
        response_format=DivergenceBrief,
    )
    chat_analysis.append(system(RECONCILIATION_PROMPT))
    chat_analysis.append(user(
        f"## Web Search Findings\n{resp_web.content}"
        f"\n\n## X Search Findings\n{resp_social.content}"
    ))
    resp_analysis = None
    for resp_analysis, chunk in chat_analysis.stream():
        pass
    if not resp_analysis or not resp_analysis.content:
        raise RuntimeError("Reconciliation returned no results")

    return DivergenceBrief.model_validate_json(resp_analysis.content)

Let's test it on a couple of different topics. Results depend on what's being discussed when you run the notebook. If outputs look thin, try a more trending topic.

In [14]:
topics = [
    "Global semiconductor supply chain shifts and chip manufacturing reshoring",
]

for topic in topics:
    print(f"\n{'=' * 60}")
    print(f"Researching: {topic}")
    print(f"{'=' * 60}")

    brief = research(topic)

    print(f"\nTopic: {brief.topic}")
    for cluster in brief.clusters:
        print(f"  {cluster.category}: {len(cluster.claims)} claims")
    print(f"\nSummary: {brief.executive_summary}\n")


Researching: Global semiconductor supply chain shifts and chip manufacturing reshoring

Topic: Semiconductor Supply Chain Shifts and Reshoring (Early 2026)
  CONSENSUS: 10 claims
  X_AHEAD_OF_PRESS: 7 claims
  PRESS_AHEAD_OF_X: 7 claims
  X_ONLY: 5 claims
  PRESS_ONLY: 4 claims

Summary: Press and X consensus on geopolitical/national security drivers, CHIPS Act subsidies, TSMC/Japan expansions, AI impacts and execution challenges for reshoring. Press is ahead with specifics on the Jan 2026 US-Taiwan trade deal, tariff percentages, investment commitments, 40% relocation target, Section 232 AI chip tariffs, US production share and Europe Chips Act 2.0 projections. X is ahead or exclusive on India/Intel/Samsung details, capacity shortages through 2027, energy disruption risks, Chip Shortage 2.0, supercycle narrative, design/IP dominance and investment opportunities, plus unique bullish sentiment with caveats.



You can also swap in different domain lists. For example, researching a developer tools topic with `TECH_DOMAINS` instead of news outlets:

In [15]:
tech_brief = research(
    "AI coding tools reshaping software development",
    web_domains=TECH_DOMAINS,
)

print(f"Topic: {tech_brief.topic}")
for cluster in tech_brief.clusters:
    print(f"\n{cluster.category} ({len(cluster.claims)} claims)")
    for claim in cluster.claims:
        print(f"  [{claim.source}] {claim.text}")
print(f"\nSummary: {tech_brief.executive_summary}")

Topic: AI Coding Tools Reshaping Software Development

CONSENSUS (10 claims)
  [Press] AI coding tools have evolved from autocomplete to agentic systems (e.g., Claude Code, Cursor)
  [X] Evolution toward agentic, full-lifecycle tools expanding beyond autocomplete to planning, code generation, testing, reviewing, and async workflows
  [Press] High/widespread adoption of AI coding tools among developers with rapid commercial success
  [X] Massive productivity gains, rapid tool adoption, and explosive growth
  [X] Shift in developer role from manual coding/writer toward orchestrator/reviewer, directing agents, high-level design, and using as thought partners/next abstraction layer
  [Press] Concerns about code quality, generic/'almost right' outputs causing bugs, debugging headaches, and technical debt
  [X] Worries about code quality, generic output, debugging headaches, and technical debt
  [Press] AI augments rather than replaces developers who adapt, with uneven benefits rather than o

## Conclusion

The same topic looks different depending on where you look. Press reporting and X discourse emphasize different claims, move at different speeds, and frame stories through different lenses. The three-pass pattern (scoped web search, then X search, then structured reconciliation) gives you a repeatable way to surface those differences.

| Design decision | Why it matters |
|---|---|
| `web_search(allowed_domains=[...])` | You control source quality instead of hoping the model picks well |
| `x_search()` | Captures real-time public discourse directly, no developer account needed |
| `include=["inline_citations"]` | Every claim is traceable, so you can verify before you trust |
| `response_format=PydanticModel` | The analysis is machine-readable, not just human-readable |

### Ideas to extend this
- **Scheduling**: Run `research()` on a cron job and track how narratives evolve over days
- **More domain lists**: Financial filings (sec.gov), academic papers (arxiv.org), government sources
- **X handle filtering**: Use `x_search(allowed_x_handles=[...])` to scope social search to specific voices
- **Date ranges**: Use `from_date` and `to_date` on `x_search()` to focus on a specific time window
- **Dashboard**: Feed the structured output into a Streamlit or Gradio app for a live research dashboard

We opened with three questions: what does the press say, what does X say, and where do they diverge? The `DivergenceBrief` schema is a structured answer to all three.